# Train an LLM to Play 2048 with Unsloth + OpenEnv

This notebook trains a language model to generate Python strategies for the 2048 game using **GRPO** (Group Relative Policy Optimization) with:
- **Unsloth** for efficient LoRA fine-tuning
- **OpenEnv** for the 2048 game environment
- **TRL** for the GRPO training loop

The model learns to output Python functions that play 2048 by receiving reward signals from actually executing the strategies against our live environment.

**Environment:** https://huggingface.co/spaces/thaihipster/game-2048-env

## 1. Install Dependencies

In [ ]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy; get_numpy = f"numpy=={numpy.__version__}"
    except: get_numpy = "numpy"
    !uv pip install -qqq \
        "torch>=2.8.0" "triton>=3.4.0" {get_numpy} torchvision bitsandbytes "transformers==4.56.2" trackio \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth" \
        git+https://github.com/triton-lang/triton.git@0add68262ab0a2e33b84524346cb27cbb2787356#subdirectory=python/triton_kernels
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth trackio
!uv pip install --upgrade --no-deps transformers==4.56.2 tokenizers trl==0.22.2 unsloth unsloth_zoo

In [ ]:
%%capture
# Install our 2048 OpenEnv environment client
!pip install git+https://huggingface.co/spaces/thaihipster/game-2048-env
!pip install fastapi uvicorn requests

## 2. Load Model with Unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 768
lora_rank = 4

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/gpt-oss-20b",
    load_in_4bit=True,
    max_seq_length=max_seq_length,
    offload_embedding=True,
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=lora_rank,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=lora_rank * 2,
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

## 3. Connect to the 2048 Environment

We connect to our live OpenEnv 2048 environment hosted on HuggingFace Spaces.

In [ ]:
from game_2048_env import Game2048Env, Game2048Action

OPENENV_URL = "https://thaihipster-game-2048-env.hf.space"

# Test connection
env = Game2048Env(base_url=OPENENV_URL)
env.connect()
result = env.reset()
print("Connected! Initial board:")
print(result.observation.board_text)
print(f"Legal actions: {result.observation.legal_actions}")
env.close()

## 4. Game Helpers

Utility functions for executing strategies against the environment and rendering boards.

In [ ]:
from unsloth import execute_with_time_limit, check_python_modules, create_locked_down_function
from typing import Callable
import itertools

def execute_strategy_against_env(strategy, max_steps=500):
    """Run a strategy function against the live 2048 environment.
    Returns (steps, max_tile, score, won)."""
    env = Game2048Env(base_url=OPENENV_URL)
    env.connect()
    result = env.reset()

    steps = 0
    while not result.observation.done and steps < max_steps:
        board = result.observation.board
        try:
            action = strategy(board)
            action = int(action)
        except:
            break
        if action not in result.observation.legal_actions:
            break
        result = env.step(Game2048Action(action=action))
        steps += 1

    max_tile = result.observation.max_tile
    score = result.observation.score
    won = result.observation.metadata.get("won", False)
    env.close()
    return steps, max_tile, score, won

@execute_with_time_limit(5)
def safe_execute_strategy(strategy):
    return execute_strategy_against_env(strategy)

In [ ]:
# Test with a simple "always move left" strategy
def always_left(board):
    return 2  # LEFT

steps, max_tile, score, won = execute_strategy_against_env(always_left)
print(f"Always-left strategy: {steps} steps, max_tile={max_tile}, score={score}, won={won}")

## 5. Prompt & Response Parsing

The model generates Python `strategy(board)` functions that return an action (0-3).

In [ ]:
prompt = """
Create a new short 2048 strategy using only native Python code.
You are given a list of list of numbers for the current board state (4x4 grid).
Each cell is 0 (empty) or a power of 2 (2, 4, 8, ..., 2048).
Output one action: 0=up, 1=down, 2=left, 3=right for the optimal next move.
Output your new short function in backticks using the format below:
```python
def strategy(board):
    return 0 # Example: move up
```
All helper functions should be inside def strategy. Only output the short function `strategy`.
""".strip()
print(prompt)

In [ ]:
def extract_function(text):
    """Extract a strategy function from model output."""
    if text.count("```") >= 2:
        first = text.find("```") + 3
        second = text.find("```", first)
        fx = text[first:second].strip()
        fx = fx.removeprefix("python\n")
        fx = fx[fx.find("def"):]
        if fx.startswith("def strategy(board):"):
            return fx
    return None

# Test extraction
sample = '```python\ndef strategy(board):\n    return 0\n```'
print(extract_function(sample))

## 6. Reward Functions

Three reward signals for GRPO:
1. **function_works** - Does the generated code parse and create a valid function?
2. **no_cheating** - Does it only use standard Python (no numpy, etc.)?
3. **strategy_succeeds** - How well does it actually play 2048?

In [ ]:
def function_works(completions, **kwargs):
    """Reward: does the generated code create a valid callable function?"""
    scores = []
    for completion in completions:
        score = 0
        response = completion[0]["content"]
        function = extract_function(response)
        if function is not None:
            ok, info = check_python_modules(function)
        if function is None or "error" in info:
            score = -2.0
        else:
            try:
                create_locked_down_function(function)
                score = 1.0
            except:
                score = -0.5
        scores.append(score)
    return scores


def no_cheating(completions, **kwargs):
    """Reward: only use standard Python imports (no numpy, etc.)."""
    scores = []
    for completion in completions:
        response = completion[0]["content"]
        function = extract_function(response)
        if function is not None:
            ok, info = check_python_modules(function)
            scores.append(1.0 if ok else -20.0)
        else:
            scores.append(-1.0)
    return scores


PRINTER = 0
def strategy_succeeds(completions, **kwargs):
    """Reward: run the strategy against the live 2048 env and score based on performance."""
    global PRINTER
    scores = []
    for completion in completions:
        response = completion[0]["content"]
        function = extract_function(response)

        should_print = PRINTER % 5 == 0
        PRINTER += 1

        if function is None:
            scores.append(0)
            continue

        ok, info = check_python_modules(function)
        if "error" in info:
            scores.append(0)
            continue

        try:
            new_strategy = create_locked_down_function(function)
        except:
            scores.append(0)
            continue

        try:
            steps, max_tile, game_score, won = safe_execute_strategy(new_strategy)
            if should_print:
                print(f"Steps={steps}, MaxTile={max_tile}, Score={game_score}, Won={won}")
                print(function[:200])

            if won:
                scores.append(20.0)  # Massive reward for winning!
            elif max_tile >= 512:
                scores.append(10.0)
            elif max_tile >= 256:
                scores.append(5.0)
            elif max_tile >= 128:
                scores.append(3.0)
            else:
                scores.append(2.0)  # At least the function ran
        except TimeoutError:
            scores.append(-1.0)
        except Exception as e:
            if should_print:
                print(f"Exception: {e}")
            scores.append(-3.0)
    return scores

## 7. Dataset & GRPO Training

In [ ]:
from datasets import Dataset

dataset = Dataset.from_list([
    {
        "prompt": [{"role": "user", "content": prompt.strip()}],
        "answer": 0,
        "reasoning_effort": "low",
    }
] * 1000)

maximum_length = len(tokenizer.apply_chat_template(
    [{"role": "user", "content": prompt.strip()}],
    add_generation_prompt=True,
))
print(f"Prompt token length: {maximum_length}")
print(f"Dataset size: {len(dataset)}")

In [ ]:
from trl import GRPOConfig, GRPOTrainer

max_prompt_length = maximum_length + 1
max_completion_length = max_seq_length - max_prompt_length

training_args = GRPOConfig(
    temperature=1.0,
    learning_rate=2e-4,
    weight_decay=0.001,
    warmup_ratio=0.1,
    lr_scheduler_type="linear",
    optim="adamw_8bit",
    logging_steps=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,  # Increase to 4 for smoother training
    num_generations=2,              # Decrease if out of memory
    max_prompt_length=max_prompt_length,
    max_completion_length=max_completion_length,
    max_steps=600,
    save_steps=100,
    report_to="trackio",
    output_dir="outputs",
)

In [ ]:
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[
        function_works,
        no_cheating,
        strategy_succeeds,
    ],
    args=training_args,
    train_dataset=dataset,
)

## 8. Train!

In [ ]:
trainer.train()

## 9. Test the Trained Model

In [ ]:
text = tokenizer.apply_chat_template(
    [{"role": "user", "content": prompt}],
    tokenize=False,
    add_generation_prompt=True,
    reasoning_effort="low",
)

from transformers import TextStreamer
_ = model.generate(
    **tokenizer(text, return_tensors="pt").to("cuda"),
    temperature=1.0,
    max_new_tokens=1024,
    streamer=TextStreamer(tokenizer, skip_prompt=False),
)

## 10. Save Model (Optional)

In [ ]:
# Save locally in mxfp4 4bit format
if False:
    model.save_pretrained_merged("finetuned_2048_model", tokenizer, save_method="mxfp4")

# Push to HuggingFace Hub
if False:
    model.push_to_hub_merged(
        "thaihipster/gpt-oss-2048-finetune",
        tokenizer,
        save_method="merged_16bit",
        token="YOUR_HF_TOKEN",
    )